# 01_07_r2_cft_id_checks

Read-only тетрадка для диагностики `ods.scd1_z_r2_ip_merchants`.

Что проверяем:
1. Сколько строк на каждый `cft_id` (ожидание уникальности / дубликаты).
2. Где именно есть дубликаты (`ALL` и `active` без `ods_deleted_flg='1'`).
3. История по выбранным `cft_id` для выбора правильного ранжирования в секции 06.
4. Проверка joined-полей (`ogrn`, `vsp_name`, `vsp_code`, `filial_rf_raw`, `tariff_name_legacy`) для тех же `cft_id`.

Ничего не удаляет и не создает в Озере.

In [ ]:
from getpass import getpass

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Конфиг
report_month_end = '2026-05-31'  # при необходимости поменяйте месяц

r2_table = 'ods.scd1_z_r2_ip_merchants'
corp_table = 'ods.scd1_z_cl_corp'
dep_table = 'ods.scd1_z_depart'
branch_table = 'ods.scd1_z_branch'
tariff_plan_table = 'ods.scd1_z_r2_tariff_plan'

print('report_month_end =', report_month_end)
print('r2_table =', r2_table)

In [ ]:
# Подключение к Impala (read-only)
if 'imp' in globals() and imp is not None:
    print('Using existing imp connection from current session')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'}
    )

try:
    imp._init_connection()
except Exception:
    pass

print('Impala connection initialized')

In [ ]:
# 0) Схема r2-таблицы: ищем поля для корректного ранжирования (valid/update/date)
with imp:
    r2_schema_df = imp.fetch(f"describe {r2_table}")

print('Columns in', r2_table)
display(r2_schema_df)

if r2_schema_df is not None and len(r2_schema_df):
    name_col = r2_schema_df.columns[0]
    cols_df = r2_schema_df.copy()
    cols_df['col_name_norm'] = cols_df[name_col].astype(str).str.strip().str.lower()

    candidate_rank_cols_df = cols_df[
        cols_df['col_name_norm'].str.contains(
            'valid|date|from|to|upd|update|create|insert|eff|begin|end|timestamp|_ts',
            regex=True,
            na=False,
        )
    ]
    print('Candidate ranking columns (date/version-like):')
    display(candidate_rank_cols_df)
else:
    print('DESCRIBE returned empty result')

In [ ]:
# 1) Базовая проверка: сколько записей приходится на каждый cft_id
sql_rows_per_cft = f"""
select
  cast(m.c_cl_org as string) as cft_id,
  count(*) as rows_cnt_all,
  sum(case when coalesce(m.ods_deleted_flg, '0') <> '1' then 1 else 0 end) as rows_cnt_active
from {r2_table} m
where m.c_cl_org is not null
group by cast(m.c_cl_org as string)
order by rows_cnt_all desc, cft_id
"""

with imp:
    rows_per_cft_df = imp.fetch(sql_rows_per_cft)

if rows_per_cft_df is None or rows_per_cft_df.empty:
    raise RuntimeError('No data returned for cft_id cardinality check')

rows_per_cft_df['rows_cnt_all'] = pd.to_numeric(rows_per_cft_df['rows_cnt_all'], errors='coerce').fillna(0).astype(int)
rows_per_cft_df['rows_cnt_active'] = pd.to_numeric(rows_per_cft_df['rows_cnt_active'], errors='coerce').fillna(0).astype(int)

cft_distinct = int(rows_per_cft_df['cft_id'].nunique())
cft_with_dups_all = int((rows_per_cft_df['rows_cnt_all'] > 1).sum())
cft_with_dups_active = int((rows_per_cft_df['rows_cnt_active'] > 1).sum())
max_rows_all = int(rows_per_cft_df['rows_cnt_all'].max())
max_rows_active = int(rows_per_cft_df['rows_cnt_active'].max())

summary_df = pd.DataFrame([
    {'metric': 'distinct_cft_id', 'value': cft_distinct},
    {'metric': 'cft_id_with_duplicates_all', 'value': cft_with_dups_all},
    {'metric': 'cft_id_with_duplicates_active', 'value': cft_with_dups_active},
    {'metric': 'max_rows_per_cft_all', 'value': max_rows_all},
    {'metric': 'max_rows_per_cft_active', 'value': max_rows_active},
])

print('Summary:')
display(summary_df)

print('\nTop 200 cft_id by rows_cnt_all:')
display(rows_per_cft_df.head(200))

dups_all_df = rows_per_cft_df[rows_per_cft_df['rows_cnt_all'] > 1].copy()
dups_active_df = rows_per_cft_df[rows_per_cft_df['rows_cnt_active'] > 1].copy()

print('\nTop 200 duplicated cft_id (ALL rows):')
display(dups_all_df.head(200))

print('\nTop 200 duplicated cft_id (ACTIVE rows only):')
display(dups_active_df.head(200))

In [ ]:
# 2) Распределение количества строк на cft_id
rows_distribution_df = (
    rows_per_cft_df
    .groupby('rows_cnt_all', as_index=False)
    .agg(cft_id_cnt=('cft_id', 'nunique'))
    .sort_values('rows_cnt_all')
)

rows_distribution_active_df = (
    rows_per_cft_df
    .groupby('rows_cnt_active', as_index=False)
    .agg(cft_id_cnt=('cft_id', 'nunique'))
    .sort_values('rows_cnt_active')
)

print('Distribution by rows_cnt_all:')
display(rows_distribution_df.head(50))

print('\nDistribution by rows_cnt_active:')
display(rows_distribution_active_df.head(50))

In [ ]:
# 3) История по выбранным cft_id (для выбора ранжирования)
# Вставьте сюда свои cft_id вручную при необходимости.
selected_cft_ids = []

if not selected_cft_ids:
    if 'dups_active_df' in globals() and len(dups_active_df):
        selected_cft_ids = dups_active_df['cft_id'].head(5).astype(str).tolist()
    elif 'dups_all_df' in globals() and len(dups_all_df):
        selected_cft_ids = dups_all_df['cft_id'].head(5).astype(str).tolist()

if not selected_cft_ids:
    raise RuntimeError('Не найдено cft_id для истории. Укажите selected_cft_ids вручную.')

print('selected_cft_ids =', selected_cft_ids)

cft_sql_list = ', '.join([f"'{x}'" for x in selected_cft_ids])

sql_history = f"""
with hist as (
  select
    cast(m.c_cl_org as string) as cft_id,
    cast(m.id as string) as r2_id,
    coalesce(m.ods_deleted_flg, '0') as ods_deleted_flg_norm,
    m.*,
    row_number() over (
      partition by cast(m.c_cl_org as string)
      order by cast(m.id as string) desc
    ) as rn_desc_by_id
  from {r2_table} m
  where cast(m.c_cl_org as string) in ({cft_sql_list})
)
select *
from hist
order by cft_id, rn_desc_by_id
"""

with imp:
    r2_history_df = imp.fetch(sql_history)

print('R2 history for selected cft_id:')
display(r2_history_df)

In [ ]:
# 4) "Рабочие" joined-поля, которые использует секция 06
sql_joined = f"""
select
  cast(m.id as string) as r2_id,
  cast(m.c_cl_org as string) as cft_id,
  cast(m.c_depart as string) as c_depart,
  cast(m.c_tariff_plan as string) as c_tariff_plan,
  coalesce(m.ods_deleted_flg, '0') as ods_deleted_flg,
  cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
  cast(dep.c_name as string) as vsp_name,
  cast(dep.c_num as string) as vsp_code,
  cast(br.c_shortlabel as string) as filial_rf_raw,
  cast(tp.c_name as string) as tariff_name_legacy
from {r2_table} m
left join {corp_table} corp
  on cast(corp.id as string) = cast(m.c_cl_org as string)
left join {dep_table} dep
  on cast(dep.id as string) = cast(m.c_depart as string)
left join {branch_table} br
  on cast(br.id as string) = cast(dep.c_filial as string)
left join {tariff_plan_table} tp
  on cast(tp.id as string) = cast(m.c_tariff_plan as string)
where cast(m.c_cl_org as string) in ({cft_sql_list})
order by cft_id, r2_id desc
"""

with imp:
    r2_joined_df = imp.fetch(sql_joined)

print('Joined attributes for selected cft_id:')
display(r2_joined_df)

## Что прислать в чат после запуска

1. `summary_df` (особенно `cft_id_with_duplicates_active` и `max_rows_per_cft_active`).
2. 10-20 строк из `dups_active_df`.
3. `r2_history_df` и `r2_joined_df` для 3-5 `cft_id`.
4. Какой `report_month_end` используем для правила "актуальна на конец месяца".

После этого можно зафиксировать точный `ORDER BY`/фильтр в секции 06.

## Диагностика узкого места секции 06

Ниже блок проверок, который помогает отделить:
1. `queue/admission` проблему кластера,
2. тяжесть выборки `r2_raw`,
3. тяжесть join-ов,
4. тяжесть ранжирования `row_number`.

Блок read-only: только `SELECT`/`EXPLAIN`.

In [ ]:
# 5) Конфиг диагностики 06 + helper-ы
import time

# Можно вставить вручную cft_id проблемного чанка из Impala UI.
# Если пусто, берем selected_cft_ids из предыдущей ячейки.
diag_cft_ids = []

if not diag_cft_ids and 'selected_cft_ids' in globals() and selected_cft_ids:
    diag_cft_ids = [str(x).strip() for x in selected_cft_ids if str(x).strip()]

if not diag_cft_ids:
    raise RuntimeError('diag_cft_ids пустой. Укажите cft_id вручную или сначала выполните ячейку с selected_cft_ids.')

run_explain_06 = True
mem_limit_diag = '8g'

print('diag_cft_ids =', diag_cft_ids)
print('report_month_end =', report_month_end)
print('mem_limit_diag =', mem_limit_diag)


def _sql_list(values):
    vals = [str(v).strip() for v in (values or []) if str(v).strip()]
    return ', '.join([f"'{v}'" for v in vals]) if vals else "''"


def _run_timed_fetch(step_name, sql_text, mem_limit='8g', show_result=True):
    start_ts = time.perf_counter()
    status = 'ok'
    err_msg = None
    out_df = None

    try:
        with imp:
            imp.execute(f'set MEM_LIMIT={mem_limit}')
            out_df = imp.fetch(sql_text)
    except Exception as exc:
        status = 'failed'
        err_msg = f"{type(exc).__name__}: {str(exc)[:500]}"

    elapsed_sec = round(time.perf_counter() - start_ts, 2)
    print(f"[{step_name}] status={status}, elapsed_sec={elapsed_sec}")

    if err_msg:
        print(f"[{step_name}] error={err_msg}")
    elif show_result and out_df is not None:
        display(out_df)

    return {
        'step': step_name,
        'status': status,
        'elapsed_sec': elapsed_sec,
        'error': err_msg,
    }, out_df

In [ ]:
# 6) Диагноз очереди/кластера: быстрые probe перед тяжелыми SQL
probe_rows = []

for i in range(1, 4):
    rec, _ = _run_timed_fetch(
        step_name=f'probe_select_1_run_{i}',
        sql_text='select 1 as probe_ping',
        mem_limit=mem_limit_diag,
        show_result=False,
    )
    probe_rows.append(rec)

probe_df = pd.DataFrame(probe_rows)
print('Queue/cluster probe:')
display(probe_df)

print('Подсказка: если probe часто > 10-15 сек, проблема скорее в кластере/очереди.')

In [ ]:
# 7) Профиль выбранного scope: объем и вариативность по cft_id
cft_sql_list_diag = _sql_list(diag_cft_ids)

sql_scope_profile = f"""
select
  cast(m.c_cl_org as string) as cft_id,
  count(*) as rows_all,
  sum(case when coalesce(m.ods_deleted_flg, '0') <> '1' then 1 else 0 end) as rows_active,
  count(distinct cast(m.id as string)) as id_versions,
  count(distinct cast(m.c_tariff_plan as string)) as tariff_versions,
  count(distinct cast(m.c_depart as string)) as depart_versions,
  min(cast(m.ods_commit_ts as string)) as min_ods_commit_ts,
  max(cast(m.ods_commit_ts as string)) as max_ods_commit_ts,
  min(cast(m.ods_insert_ts as string)) as min_ods_insert_ts,
  max(cast(m.ods_insert_ts as string)) as max_ods_insert_ts
from {r2_table} m
where cast(m.c_cl_org as string) in ({cft_sql_list_diag})
group by cast(m.c_cl_org as string)
order by rows_all desc, cft_id
"""

_, scope_profile_df = _run_timed_fetch(
    step_name='scope_profile_per_cft',
    sql_text=sql_scope_profile,
    mem_limit=mem_limit_diag,
)

if scope_profile_df is not None and len(scope_profile_df):
    scope_profile_df['rows_all'] = pd.to_numeric(scope_profile_df['rows_all'], errors='coerce')
    print('Top heavy cft_id in current diagnostic scope:')
    display(scope_profile_df)
else:
    print('No scope profile rows returned')

In [ ]:
# 8) Этапный тайминг, как в секции 06: raw -> join -> ranked
# Это главный тест, который показывает, где именно тормозит.

sql_stage_raw = f"""
with r2_raw as (
  select
    cast(m.id as string) as r2_id,
    cast(m.c_cl_org as string) as cft_id,
    cast(m.c_depart as string) as c_depart,
    cast(m.c_tariff_plan as string) as c_tariff_plan,
    coalesce(m.ods_deleted_flg, '0') as ods_deleted_flg
  from {r2_table} m
  where cast(m.c_cl_org as string) in ({cft_sql_list_diag})
    and m.c_cl_org is not null
    and coalesce(m.ods_deleted_flg, '0') <> '1'
)
select
  count(*) as rows_raw,
  count(distinct cft_id) as cft_distinct_raw
from r2_raw
"""

sql_stage_join = f"""
with r2_raw as (
  select
    cast(m.id as string) as r2_id,
    cast(m.c_cl_org as string) as cft_id,
    cast(m.c_depart as string) as c_depart,
    cast(m.c_tariff_plan as string) as c_tariff_plan,
    coalesce(m.ods_deleted_flg, '0') as ods_deleted_flg
  from {r2_table} m
  where cast(m.c_cl_org as string) in ({cft_sql_list_diag})
    and m.c_cl_org is not null
    and coalesce(m.ods_deleted_flg, '0') <> '1'
),
r2_enriched as (
  select
    r2.r2_id,
    r2.cft_id,
    r2.c_tariff_plan,
    cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
    cast(dep.c_name as string) as vsp_name,
    cast(dep.c_num as string) as vsp_code,
    cast(br.c_shortlabel as string) as filial_rf_raw,
    cast(tp.c_name as string) as tariff_name_legacy
  from r2_raw r2
  left join {corp_table} corp on cast(corp.id as string) = r2.cft_id
  left join {dep_table} dep on cast(dep.id as string) = r2.c_depart
  left join {branch_table} br on cast(br.id as string) = cast(dep.c_filial as string)
  left join {tariff_plan_table} tp on cast(tp.id as string) = r2.c_tariff_plan
)
select
  count(*) as rows_joined,
  count(distinct cft_id) as cft_distinct_joined
from r2_enriched
"""

sql_stage_ranked = f"""
with r2_raw as (
  select
    cast(m.id as string) as r2_id,
    cast(m.c_cl_org as string) as cft_id,
    cast(m.c_depart as string) as c_depart,
    cast(m.c_tariff_plan as string) as c_tariff_plan,
    coalesce(m.ods_deleted_flg, '0') as ods_deleted_flg
  from {r2_table} m
  where cast(m.c_cl_org as string) in ({cft_sql_list_diag})
    and m.c_cl_org is not null
    and coalesce(m.ods_deleted_flg, '0') <> '1'
),
r2_enriched as (
  select
    r2.r2_id,
    r2.cft_id,
    r2.c_tariff_plan,
    cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
    cast(dep.c_name as string) as vsp_name,
    cast(dep.c_num as string) as vsp_code,
    cast(br.c_shortlabel as string) as filial_rf_raw,
    cast(tp.c_name as string) as tariff_name_legacy
  from r2_raw r2
  left join {corp_table} corp on cast(corp.id as string) = r2.cft_id
  left join {dep_table} dep on cast(dep.id as string) = r2.c_depart
  left join {branch_table} br on cast(br.id as string) = cast(dep.c_filial as string)
  left join {tariff_plan_table} tp on cast(tp.id as string) = r2.c_tariff_plan
),
r2_ranked as (
  select
    e.*,
    row_number() over (
      partition by e.cft_id
      order by
        case when e.filial_rf_raw is not null and trim(e.filial_rf_raw) <> '' then 0 else 1 end,
        case when e.vsp_name is not null and trim(e.vsp_name) <> '' then 0 else 1 end,
        case when e.vsp_code is not null and trim(e.vsp_code) <> '' then 0 else 1 end,
        case when e.tariff_name_legacy is not null and trim(e.tariff_name_legacy) <> '' then 0 else 1 end,
        e.r2_id desc
    ) as rn
  from r2_enriched e
)
select
  count(*) as rows_ranked_kept,
  count(distinct cft_id) as cft_distinct_ranked
from r2_ranked
where rn = 1
"""

stage_records = []
for step_name, sql_text in [
    ('stage_raw_only', sql_stage_raw),
    ('stage_join_enrich', sql_stage_join),
    ('stage_rank_row_number', sql_stage_ranked),
]:
    rec, _ = _run_timed_fetch(
        step_name=step_name,
        sql_text=sql_text,
        mem_limit=mem_limit_diag,
    )
    stage_records.append(rec)

stage_timing_df = pd.DataFrame(stage_records)
print('06 diagnostics stage timing:')
display(stage_timing_df)

print('Интерпретация:')
print('- stage_raw_only долгий -> проблема на базовом чтении/filter')
print('- stage_join_enrich резко хуже raw -> bottleneck в join/cast')
print('- stage_rank_row_number резко хуже join -> bottleneck в оконке/ranking')

In [ ]:
# 9) Опционально: EXPLAIN по этапам (помогает понять, где тяжелый план)
if run_explain_06:
    explain_queries = {
        'explain_stage_raw_only': f"explain {sql_stage_raw}",
        'explain_stage_join_enrich': f"explain {sql_stage_join}",
        'explain_stage_rank_row_number': f"explain {sql_stage_ranked}",
    }

    explain_rows = []
    for step_name, ex_sql in explain_queries.items():
        rec, explain_df = _run_timed_fetch(
            step_name=step_name,
            sql_text=ex_sql,
            mem_limit=mem_limit_diag,
            show_result=False,
        )
        explain_rows.append(rec)

        print(f"\n{step_name}: first 80 lines")
        if explain_df is not None and len(explain_df):
            display(explain_df.head(80))
        else:
            print('No explain output')

    explain_timing_df = pd.DataFrame(explain_rows)
    print('\nEXPLAIN timing:')
    display(explain_timing_df)
else:
    print('run_explain_06=False -> skip explain block')

## Что смотреть в результатах блока диагностики 06

1. `probe_df`:
   - если probe > 10-15 сек, проблема вероятнее в кластере/очереди.
2. `scope_profile_per_cft`:
   - покажет, нет ли перекоса (очень тяжелого `cft_id`) в текущем scope.
3. `stage_timing_df`:
   - где скачок времени (`raw`, `join`, `rank`).
4. `EXPLAIN`:
   - используется для сравнения относительной сложности этапов.

Пришлите `probe_df`, `scope_profile_per_cft`, `stage_timing_df` и (по возможности) первые строки `EXPLAIN` для `stage_join_enrich`/`stage_rank_row_number`.